# Morning Goal Model - Multitask Training

This notebook provides a complete pipeline for training the Multitask BERT model on Google Colab.

## Tasks
1. **Topic Classification**: 16 classes
2. **Sentiment Analysis**: 3 classes (Negative, Neutral, Positive)

## Prerequisites
- Upload the `MorningGoalModel` project folder to your Google Drive.
- Mount Google Drive.


In [ ]:
# @title 1. Install Dependencies
!pip install transformers datasets accelerate scikit-learn pandas coremltools

In [ ]:
# @title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 3. Setup Project Path
import sys
import os

# Set this to the path where you uploaded the project
PROJECT_PATH = '/content/drive/MyDrive/MorningGoalModel'

if os.path.exists(PROJECT_PATH):
    os.chdir(PROJECT_PATH)
    sys.path.append(PROJECT_PATH)
    print(f"Current working directory: {os.getcwd()}")
else:
    print(f"Error: Path {PROJECT_PATH} does not exist. Please check your Drive structure.")

In [ ]:
# @title 4. Import Training Module
try:
    from src.training.train_multitask import run_training
    print("Successfully imported training module.")
except ImportError as e:
    print(f"Import Error: {e}")
    print("Make sure you are in the correct directory and 'src' is importable.")

In [ ]:
# @title 5. Configure Training Arguments
class TrainingArgs:
    def __init__(self):
        self.data_dir = "data/processed"
        # Use a standard BERT model if the distilled student is not available
        self.base_model = "bert-base-chinese" 
        # Or use local path if you have uploaded the trained models
        # self.base_model = "models/trained/distill_student"
        
        self.batch_size = 32
        self.epochs = 5
        self.lr = 2e-5
        self.max_length = 128
        self.output_dir = "models/trained/multitask_model_colab"
        self.seed = 42
        self.limit_train = 0  # 0 means use all data
        self.limit_eval = 0   # 0 means use all data
        self.grad_accum = 1
        self.num_topic_labels = 16
        self.num_sentiment_labels = 3

args = TrainingArgs()
print("Configuration ready.")

In [ ]:
# @title 6. Run Training
# Ensure data exists
if not os.path.exists(os.path.join(args.data_dir, "train_multitask.csv")):
    print("Error: Training data not found. Please ensure 'data/processed' contains the multitask CSV files.")
else:
    print("Starting training... this may take a while.")
    run_training(args)

## 7. Inference & Testing
Use the trained model to make predictions on new text.

In [ ]:
# @title 7.1 Run Inference
from src.evaluation.multitask_inference import run_inference

# Define some test sentences
test_texts = [
    "今天完成了5公里跑步，感觉非常棒！",
    "最近工作压力好大，经常加班到深夜。",
    "这周末打算去图书馆看书学习。",
    "为了买房正在努力存钱。",
    "和朋友吵架了，心情很低落。"
]

# Path to the trained model
model_path = args.output_dir

if os.path.exists(model_path):
    print("Running inference...")
    results = run_inference(model_path, test_texts)
else:
    print(f"Error: Model path {model_path} does not exist. Did you run training?")

In [ ]:
# @title 7.2 Detailed Evaluation Report
from src.evaluation.evaluate_multitask import evaluate_model

test_data_path = os.path.join(args.data_dir, "test_multitask.csv")
model_path = args.output_dir
eval_output_dir = "evaluation_results"

if os.path.exists(model_path) and os.path.exists(test_data_path):
    print("Starting comprehensive evaluation...")
    evaluate_model(model_path, test_data_path, eval_output_dir)
    print(f"Check the report in {eval_output_dir}/evaluation_report.md")
else:
    print("Error: Model or test data not found.")

## 8. Export to CoreML
Export the trained model to `.mlpackage` format for iOS deployment.

In [ ]:
# @title 8.1 Export Model
from src.export.export_multitask_coreml import export_to_coreml

model_path = args.output_dir
output_path = "models/coreml/multitask_model.mlpackage"

if os.path.exists(model_path):
    print(f"Exporting model from {model_path} to {output_path}...")
    export_to_coreml(model_path, output_path)
    print("Export complete. You can download the .mlpackage from the file browser.")
else:
    print("Error: Model path does not exist.")